# Create Model from Training Run for Batch
This notebook enables running one-off batch inferenc jobs. The user will provide a training job name and, optionally, a pointer to the checkpoints weights to use for inference. The user will then provide a customer inference file that contains the specific outputs the user desires. The notebook will create a SageMaker Model for the run. 

The second part of the notebook will take in the input and output parameters for running the batch job.

## Model Parameters
Please fill in the model parameters below

In [ ]:
# Provide a unique job name for this batch inference run.
# The job name will be used to name the created model and name the batch job
JOB_NAME = ""

# Name of the SageMaker Training job used to train the model
sm_training_job_name = ""

# (OPTIONAL) S3 URI to the checkpoint weights 
# Leave blank to use the best weights as computed during testing during the training run
# NOTE!!! If the training job did not "Complete" (e.g., was "Stopped" or "Failed"), you must provide
# a checkpoint to use for the weights.
model_weights_s3_uri = ""

# (OPTIONAL) Inference Container
# Can optionally point to the pytorch inference container to use for the model hosted in ECR.
# If none provided, will use the SageMaker hosted container with same PyTorch and Python versions
# as used during training.
inference_container_uri = ""

# The SageMaker role to use, if non-provided will use the default SageMaker execution role
role = ""


## Inference File
Create the inference file to be used for this batch job.

Copy the template inference file `src/batch_inference_template.py` and give the file a unique name. 
Ensure that the new file is also placed in the `src` directory.

The bulk of the logic will be updating the `predict_fn()` and `output_fn()` to perform the desired inference and output the desired content.

If parameters need to be passed for additional calculationgs, we recommend setting them as environment variable in the cell below. Load these variables in the `model_fn()` and store them in the `data_param` dict already created by this function. Note, these parameters will be shared by all instances of the model. Per image parameters should be passed in as input.


In [ ]:
# The name of the inference file here. Ensure the file is located under the `src` directory in this repo
inference_filename = ""

# Update to point to the `src` folder in this directory
src_directory = "../src"

# Add any environment variables you want to pass to the model here
# Can also override default values used when making models here as well
additional_environment_variables = {
    # "EXAMPLE_VARIABLE": "VALUE",
}

# Optional Parameters
Below are some optional model parameters you can set if desired.

Please refer to SageMaker documentation on what these values provide.

In [ ]:
# Some default model variables
max_payload_size_mb=25
workers_per_model = 1
env_variables = {
    # Update the maximum payload size for TS. Needs to match SageMaker's max payload size
    # Default for TS is 6,553,500 bytes
    # Convert max_payload_size in MB to Bytes
    "TS_DEFAULT_WORKERS_PER_MODEL": str(workers_per_model),
    "TS_MAX_REQUEST_SIZE": str(max_payload_size_mb * 1024 * 1024)
}

pytorch_version = "2.3"
inference_target_instance = "ml.m5.2xlarge"

# Provide the uri for the inference image directly here, otherwise will use SageMaker hosted container
# based on pytorch version.
image_uri = ""

# Build Model
Do not modify unless needed.

Builds the SageMaker model with the following steps:
1. Get training job information
2. Get 
3. 

In [ ]:
import torch
from tarfile import TarFile, TarInfo
import boto3
import sagemaker
from sagemaker.pytorch import PyTorchModel
from pathlib import Path
import shutil

In [ ]:
# Some helper variables
job_directory = Path(f"model_creation_{JOB_NAME}")

orig_model_tar_file = job_directory / "orig_model.tar.gz"
model_weights_directory = job_directory / "model"
model_source_directory = job_directory / "source"

model_tar_file = job_directory / "model.tar.gz"
source_tar_file = job_directory / "source.tar.gz"

if job_directory.exists():
    raise ValueError("Folder with job name already used. Please use a different job name or clean up this folder to proceed")

In [ ]:
s3_client = boto3.client("s3")
sm_client = boto3.client("sagemaker")

In [ ]:
# Get the model.tar.gz file associated with the training job if provided
training_job = sm_client.describe_training_job(TrainingJobName=sm_training_job_name)
model_s3_uri = training_job["ModelArtifacts"]["S3ModelArtifacts"]
bucket, key = model_s3_uri.replace("s3://", "").split("/", 1)
s3_client.download_file(bucket, key, orig_model_tar_file)
with TarFile.open(orig_model_tar_file, "r:gz") as tf:
    tf.extractall(model_weights_directory)

if (model_weights_directory / "code").exists:
    (model_weights_directory / "code").rmdir()

In [ ]:
# If checkpoints weight is given, download the weights
if model_weights_s3_uri:
    bucket, key = model_weights_s3_uri.replace("s3://", "").split("/", 1)
    extension = Path(key).suffix
    s3_client.download_file(bucket, key, model_weights_directory / f"best{extension}")

In [ ]:
# Copy over source code and inference filename
model_source_directory.mkdir()
shutil.copy(src_directory, model_source_directory)
if not (model_source_directory / inference_filename).exists():
    raise ValueError(f"Could not find '{inference_filename}' in source directory '{src_directory}'")

In [ ]:
# Tar the model and source directories and place into S3
# Create new tarfile
def filter_temp_dirs(x: TarInfo):
    if "ipynb_checkpoints" in x.name or "__pycache__" in x.name:
        return None
    return x

# Create tarfile with just model weights
with TarFile.open(model_tar_file, "w:gz") as model_tar:
    model_tar.add(model_weights_directory, arcname="", filter=filter_temp_dirs)
    print("Model Tar Contents: ")
    print(model_tar.list())
    

# Create tarfile with just the source code
with TarFile.open(source_tar_file, "w:gz") as model_tar:
    model_tar.add(model_source_directory, arcname="", filter=filter_temp_dirs)
    print("Source Tar Contents: ")
    print(model_tar.list())

In [ ]:
bucket = sagemaker.Session().default_bucket()
# Upload the model tar file
s3_client.upload_file(model_tar_file, bucket, model_tar_file)

# Upload the source tar file
s3_client.upload_file(source_tar_file, bucket, source_tar_file)

# Create the S3 URIs for each
model_tar_uri = f"s3://{bucket}/{model_tar_file}"
source_tar_uri = f"s3://{bucket}/{source_tar_file}"

In [ ]:
# Get the execution role arn if not provided above
role = role or sagemaker.get_execution_role()
if not image_uri:
    image_uri = sagemaker.image_uris.retrieve(framework="PyTorch", region='us-west-2', version=pytorch_version, instance_type=inference_target_instance)

# Create the model
model = PyTorchModel(
    name=JOB_NAME,
    image_uri=image_uri,
    model_data=model_tar_uri,
    role=role or sagemaker.get_execution_role(),
    source_dir=source_tar_uri,
    entry_point=inference_filename,
    env=env_variables,
    sagemaker_session=sagemaker.Session(),
    model_server_workers=workers_per_model
)

model.create()

# Sample Image Pair Input
The next section goes over some example data to create a batch job to process an image and mask pair within the model.

## Create Input
A transform job can take in a single s3 object to process within the `input_fn`. To allow passing in pairs of items, we will do a small work around of creating JSON objects to point to these image/mask pairs for the model to load.

This block
1. Finds s3 uris to all images and masks
2. Create json files for each pair
3. Upload files to S3
4. Create manifest file from uploaded json files

### Get Image and Mask URIs

In [ ]:
image_s3_uri_prefix = "" # e.g."s3://bucket/path/to/images/"
mask_s3_uri_prefix = ""

In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

def list_s3_files(s3_uri: str, region: str = "us-west-2"):
    s3 = boto3.resource("s3")
    # Split s3 uri into bucket and prefix
    bucket, prefix = s3_uri.removeprefix("s3://").split("/", 1)
    # Add ending "/" if not present on prefix
    prefix = prefix if prefix.endswith("/") else prefix + "/"

    # Get filenames for all objects on the prefix
    filenames = []
    for obj in s3.Bucket(bucket).objects.filter(Prefix=prefix):
        filename = obj.key
        if filename.endswith(IMAGE_EXTENSIONS):
            filenames.append(f"{bucket}/{filename}")

    return sorted(filenames)

In [ ]:
image_s3_uris = list_s3_files(image_s3_uri_prefix)
mask_s3_uris = list_s3_files(mask_s3_uri_prefix)

# TODO: Add some filtering to ensure image and mask contain same filenames

### Create JSON and Upload

In [ ]:
import json
from pathlib import Path
def upload_json_to_s3(s3_resource, dictionary: dict, s3_uri: str, indent=2) -> None:
    bucket, key = s3_uri.replace("s3://").split("/", 1)
    obj = s3_resource.Object(bucket, key)
    obj.put(Body=json.dumps(dictionary, indent=indent).encode())


In [ ]:
# Location to upload the json s3 files
json_s3_prefix = ""

# Ensure json_s3_prefix ends with a "/"
json_s3_prefix = json_s3_prefix if json_s3_prefix.endswith("/") else json_s3_prefix + "/"

s3 = boto3.resource("s3")

bucket, prefix = json_s3_prefix.replace("s3://").split("/", 1)

manifest_content = [{"prefix": json_s3_prefix}]

for image, mask in zip(image_s3_uris, mask_s3_uris):
    content = {"image": image, "mask": mask}
    image_name = Path(image).stem
    json_uri = f"{json_s3_prefix}{image_name}.json"
    upload_json_to_s3(s3, content, json_uri)
    manifest_content.append(f"{image_name}.json")
    
# Upload the manifest file
manifest_uri = f"{json_s3_prefix}image_mask.manifest"
upload_json_to_s3(s3, manifest_content, manifest_uri)

print(f"Created image_mask manifest file with {len(manifest_content) - 1} pairs:")
print(f"\t uri: {manifest_uri}")

## Update Input Function in Template
Update the inference template's `input_fn` to read in the json files and return bytes for 2 images

In [ ]:
JSON_CONTENT_TYPE = "application/json"

def get_s3_object(s3_resource, s3_uri: str):
    bucket, key = s3_uri.replace("s3://", "").split("/", 1)
    try:
        obj = s3_resource.Object(bucket, key).get()
        return obj["Body"].read()
    except s3_resource.meta.client.exceptions.NoSuchKey:
        return None


def input_fn(serialized_input_data, content_type=JSON_CONTENT_TYPE):
    if content_type == JSON_CONTENT_TYPE:
        content = json.loads(serialized_input_data)
        s3_resource = boto3.resource("s3")
        image = get_s3_object(s3_resource, content["image"])
        mask = get_s3_object(s3_resource, content["mask"])
        
        if image is None or mask is None:
            raise ValueError(f"Unable to load either mask or image in set: \n  {json.dumps(content, indent=2)}")
        
        return (image, mask)   
    else:
        raise ValueError(f"Unexpected content type given: {content_type}")